In [ ]:
# CELL 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# CELL 2: Imports and load saved quantile results
# No re-evaluation needed — reuses cal_results.pkl and test_results.pkl
import os, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

PKL_DIR    = '/content/drive/MyDrive/avec2014_mondrian_recal'
OCC_CSV    = '/content/drive/MyDrive/avec2014_occlusion.csv'
SAVE_DIR   = '/content/drive/MyDrive/avec2014_sparse_subgroup'
os.makedirs(SAVE_DIR, exist_ok=True)

N_QUANTILES = 99
M_BINS      = 4
ALPHA       = 0.1
TARGET      = 1 - ALPHA
Q_LO_IDX    = int(ALPHA / 2 * N_QUANTILES)
Q_HI_IDX    = int((1 - ALPHA / 2) * N_QUANTILES) - 1
MED_IDX     = N_QUANTILES // 2
K_REPS      = 200   # repetitions per N for bootstrap CI

with open(os.path.join(PKL_DIR, 'cal_results.pkl'), 'rb') as f:
    cal_results = pickle.load(f)
with open(os.path.join(PKL_DIR, 'test_results.pkl'), 'rb') as f:
    test_results = pickle.load(f)

occ_df      = pd.read_csv(OCC_CSV)
glasses_map = dict(zip(occ_df['filename'], occ_df['has_glasses']))
beard_map   = dict(zip(occ_df['filename'], occ_df['has_beard']))

print(f'Cal results: {len(cal_results)} videos')
print(f'Test results: {len(test_results)} videos')

# Count minority groups in calibration set
n_glasses_cal = sum(1 for stem, _, _ in cal_results if glasses_map.get(stem, 0) == 1)
n_beard_cal   = sum(1 for stem, _, _ in cal_results if beard_map.get(stem, 0) == 1)
print(f'Calibration set — Glasses: {n_glasses_cal}, Beard: {n_beard_cal}')

Cal results: 50 videos
Test results: 50 videos
Calibration set — Glasses: 8, Beard: 2


In [ ]:
# CELL 3: Core FUQ functions

def compute_beta(cal_results_subset):
    scores = np.array([
        max(q[Q_LO_IDX] - y, y - q[Q_HI_IDX])
        for _, y, q in cal_results_subset
    ])
    return np.quantile(scores, 1 - ALPHA)

def build_cal_df(cal_results_subset, attr_map, attr_name):
    rows = []
    for stem, y_true, q_pred in cal_results_subset:
        rows.append({
            'stem':    stem,
            'y_true':  y_true,
            'y_lo':    q_pred[Q_LO_IDX],
            'y_hi':    q_pred[Q_HI_IDX],
            'r':       max(q_pred[Q_LO_IDX] - y_true, y_true - q_pred[Q_HI_IDX]),
            attr_name: attr_map.get(stem, 0)
        })
    return pd.DataFrame(rows)

def build_bins(cal_df, m_bins=M_BINS):
    cal_sorted = cal_df.sort_values('y_true').reset_index(drop=True)
    N_cal      = len(cal_sorted)
    bin_size   = N_cal // m_bins
    bins = []
    for m in range(m_bins):
        s  = m * bin_size
        e  = (m+1)*bin_size if m < m_bins-1 else N_cal
        bd = cal_sorted.iloc[s:e]
        bins.append({'m': m, 'l': bd['y_true'].min(), 'u': bd['y_true'].max(), 'data': bd})
    return bins

def avg_coverage(bins, r_hat, group_val, attr_name):
    covs = []
    for m, b in enumerate(bins):
        g = b['data'][b['data'][attr_name] == group_val]
        if len(g) == 0:
            continue
        r = r_hat[(m, group_val)]
        c = sum(1 for _, row in g.iterrows()
                if (row['y_lo']-r) <= row['y_true'] <= (row['y_hi']+r))
        covs.append(c / len(g))
    return np.mean(covs) if covs else 0.0

def slope_down(bin_data, r, group_val, attr_name, min_size=5):
    g = bin_data[bin_data[attr_name] == group_val]
    scores = sorted(g['r'].tolist()) if len(g) >= min_size else sorted(bin_data['r'].tolist())
    n_eff  = len(g) if len(g) >= min_size else len(bin_data)
    below  = [s for s in scores if s < r]
    if not below or n_eff == 0: return 0.0
    return (1.0/n_eff) / (r - below[-1] + 1e-8)

def slope_up(bin_data, r, group_val, attr_name, min_size=5):
    g = bin_data[bin_data[attr_name] == group_val]
    scores = sorted(g['r'].tolist()) if len(g) >= min_size else sorted(bin_data['r'].tolist())
    n_eff  = len(g) if len(g) >= min_size else len(bin_data)
    above  = [s for s in scores if s > r]
    if not above or n_eff == 0: return float('inf')
    return (1.0/n_eff) / (above[0] - r + 1e-8)

def slope_down_orig(bin_data, r, group_val, attr_name, min_size=None):
    g = bin_data[bin_data[attr_name] == group_val]
    if len(g) == 0: return 0.0
    scores = sorted(g['r'].tolist())
    below  = [s for s in scores if s < r]
    if not below: return 0.0
    return (1.0/len(g)) / (r - below[-1] + 1e-8)

def slope_up_orig(bin_data, r, group_val, attr_name, min_size=None):
    g = bin_data[bin_data[attr_name] == group_val]
    if len(g) == 0: return float('inf')
    scores = sorted(g['r'].tolist())
    above  = [s for s in scores if s > r]
    if not above: return float('inf')
    return (1.0/len(g)) / (above[0] - r + 1e-8)

def fuq_optimize(bins, r_hat, groups, attr_name, sd_fn, su_fn, min_size=5, max_iter=500):
    for iteration in range(max_iter):
        cov = {g: avg_coverage(bins, r_hat, g, attr_name) for g in groups}
        if all(abs(cov[g] - TARGET) < 0.02 for g in groups):
            return r_hat, cov
        over  = max(groups, key=lambda g: cov[g])
        under = min(groups, key=lambda g: cov[g])
        if cov[over] <= TARGET:
            for g in groups:
                for m in range(M_BINS):
                    if su_fn(bins[m]['data'], r_hat[(m,g)], g, attr_name, min_size) < float('inf'):
                        r_hat[(m,g)] += 0.1
            continue
        sd_best = max(range(M_BINS), key=lambda m: sd_fn(bins[m]['data'], r_hat[(m,over)],  over,  attr_name, min_size))
        su_best = min(range(M_BINS), key=lambda m: su_fn(bins[m]['data'], r_hat[(m,under)], under, attr_name, min_size))
        gd = sd_fn(bins[sd_best]['data'], r_hat[(sd_best,over)],  over,  attr_name, min_size)
        gu = su_fn(bins[su_best]['data'], r_hat[(su_best,under)], under, attr_name, min_size)
        if gu > gd and cov[under] >= TARGET - 0.02:
            return r_hat, cov
        r_hat[(sd_best,over)]  -= 0.1
        r_hat[(su_best,under)] += 0.1
    return r_hat, {g: avg_coverage(bins, r_hat, g, attr_name) for g in groups}

def apply_intervals(test_results, bins, r_hat, attr_map, attr_name, groups):
    rows = []
    for stem, y_true, q_pred in test_results:
        y_lo = q_pred[Q_LO_IDX]
        y_hi = q_pred[Q_HI_IDX]
        g    = attr_map.get(stem, 0)
        ulo, uhi = float('inf'), float('-inf')
        for m, b in enumerate(bins):
            r  = r_hat[(m, g)]
            lo = max(y_lo-r, b['l'])
            hi = min(y_hi+r, b['u'])
            if lo <= hi:
                ulo = min(ulo, lo)
                uhi = max(uhi, hi)
        if ulo == float('inf'):
            r   = r_hat[(0, g)]
            ulo = y_lo - r
            uhi = y_hi + r
        rows.append({
            'stem': stem, 'y_true': y_true,
            attr_name: g, 'covered': ulo <= y_true <= uhi,
            'width': uhi - ulo
        })
    df = pd.DataFrame(rows)
    picp_all  = df['covered'].mean()
    mpiw_all  = df['width'].mean()
    picp_min  = df[df[attr_name]==groups[1]]['covered'].mean() if (df[attr_name]==groups[1]).any() else np.nan
    picp_maj  = df[df[attr_name]==groups[0]]['covered'].mean() if (df[attr_name]==groups[0]).any() else np.nan
    gap       = abs(picp_maj - picp_min) if not np.isnan(picp_min) else np.nan
    return picp_all, mpiw_all, picp_min, picp_maj, gap

def cqr_results(cal_subset, test_results, attr_map, attr_name, groups):
    beta = compute_beta(cal_subset)
    rows = []
    for stem, y_true, q_pred in test_results:
        lo = q_pred[Q_LO_IDX] - beta
        hi = q_pred[Q_HI_IDX] + beta
        g  = attr_map.get(stem, 0)
        rows.append({'stem': stem, 'y_true': y_true, attr_name: g,
                     'covered': lo<=y_true<=hi, 'width': hi-lo})
    df = pd.DataFrame(rows)
    picp_all = df['covered'].mean()
    mpiw_all = df['width'].mean()
    picp_min = df[df[attr_name]==groups[1]]['covered'].mean() if (df[attr_name]==groups[1]).any() else np.nan
    picp_maj = df[df[attr_name]==groups[0]]['covered'].mean() if (df[attr_name]==groups[0]).any() else np.nan
    gap      = abs(picp_maj - picp_min) if not np.isnan(picp_min) else np.nan
    return picp_all, mpiw_all, picp_min, picp_maj, gap

print('Core functions defined.')

Core functions defined.


In [ ]:
# CELL 4: Main sweep function
# For each N, subsample N minority calibration samples K times,
# run CQR / Original FUQ / Shrinkage FUQ, record metrics.

def run_sweep(cal_results, test_results, attr_map, attr_name,
              groups, n_values, k_reps=K_REPS,
              min_group_size=5, rng_seed=42):
    """
    groups[0] = majority (e.g. no glasses = 0)
    groups[1] = minority (e.g. glasses = 1)
    """
    rng = np.random.default_rng(rng_seed)

    # Split calibration into minority and majority
    cal_minority = [(s,y,q) for s,y,q in cal_results if attr_map.get(s,0)==groups[1]]
    cal_majority = [(s,y,q) for s,y,q in cal_results if attr_map.get(s,0)==groups[0]]
    max_minority  = len(cal_minority)

    print(f'Minority cal samples: {max_minority}  Majority cal samples: {len(cal_majority)}')

    results = {n: {'cqr': [], 'fuq_orig': [], 'fuq_shrunk': []} for n in n_values}

    for n in n_values:
        n_actual = min(n, max_minority)  # cap at actual minority size
        is_capped = (n > max_minority)
        label = f'N={n}' + (' (all)' if is_capped else '')
        print(f'  Running N={n} (actual={n_actual}) x {k_reps} reps...', flush=True)

        for rep in range(k_reps):
            # Subsample minority group
            if n_actual < max_minority:
                idx = rng.choice(max_minority, size=n_actual, replace=False)
                cal_min_sub = [cal_minority[i] for i in idx]
            else:
                cal_min_sub = cal_minority  # use all

            cal_sub = cal_majority + cal_min_sub

            # --- CQR ---
            metrics = cqr_results(cal_sub, test_results, attr_map, attr_name, groups)
            results[n]['cqr'].append(metrics)

            # Build bins and beta for FUQ
            beta  = compute_beta(cal_sub)
            cal_df = build_cal_df(cal_sub, attr_map, attr_name)
            bins  = build_bins(cal_df)

            # --- Original FUQ ---
            r_hat_o = {(m,g): beta for m in range(M_BINS) for g in groups}
            r_hat_o, _ = fuq_optimize(bins, r_hat_o, groups, attr_name,
                                       slope_down_orig, slope_up_orig, min_size=None)
            metrics_o = apply_intervals(test_results, bins, r_hat_o, attr_map, attr_name, groups)
            results[n]['fuq_orig'].append(metrics_o)

            # --- Shrinkage FUQ ---
            r_hat_s = {(m,g): beta for m in range(M_BINS) for g in groups}
            r_hat_s, _ = fuq_optimize(bins, r_hat_s, groups, attr_name,
                                       slope_down, slope_up, min_size=min_group_size)
            metrics_s = apply_intervals(test_results, bins, r_hat_s, attr_map, attr_name, groups)
            results[n]['fuq_shrunk'].append(metrics_s)

    return results, max_minority


def summarise_sweep(results, n_values, max_minority):
    """
    Compute mean and 95% CI for each metric across K repetitions.
    Metrics tuple: (picp_all, mpiw_all, picp_min, picp_maj, gap)
    """
    metric_names = ['picp_all', 'mpiw_all', 'picp_min', 'picp_maj', 'gap']
    summary = {}
    for n in n_values:
        summary[n] = {}
        for method in ['cqr', 'fuq_orig', 'fuq_shrunk']:
            arr = np.array(results[n][method])  # (K, 5)
            row = {}
            for i, name in enumerate(metric_names):
                vals = arr[:, i]
                vals = vals[~np.isnan(vals)]
                if len(vals) == 0:
                    row[name + '_mean'] = np.nan
                    row[name + '_lo']   = np.nan
                    row[name + '_hi']   = np.nan
                else:
                    row[name + '_mean'] = vals.mean()
                    row[name + '_lo']   = np.percentile(vals, 2.5)
                    row[name + '_hi']   = np.percentile(vals, 97.5)
            summary[n][method] = row
    return summary

print('Sweep functions defined.')

Sweep functions defined.


In [ ]:
# CELL 5: Run sweep for GLASSES
print('=' * 60)
print('EXPERIMENT 1: GLASSES subgroup calibration size sweep')
print(f'K={K_REPS} repetitions per N, MIN_GROUP_SIZE=5')
print('=' * 60)

N_VALUES_GLASSES = [2, 3, 4, 5, 7, 10, 15, 20]
GROUPS_GLASSES   = [0, 1]   # 0=no glasses, 1=glasses

glasses_results, max_glasses = run_sweep(
    cal_results, test_results, glasses_map, 'has_glasses',
    GROUPS_GLASSES, N_VALUES_GLASSES,
    k_reps=K_REPS, min_group_size=5
)
glasses_summary = summarise_sweep(glasses_results, N_VALUES_GLASSES, max_glasses)
print('Glasses sweep complete.')

EXPERIMENT 1: GLASSES subgroup calibration size sweep
K=200 repetitions per N, MIN_GROUP_SIZE=5
Minority cal samples: 8  Majority cal samples: 42
  Running N=2 (actual=2) x 200 reps...
  Running N=3 (actual=3) x 200 reps...
  Running N=4 (actual=4) x 200 reps...
  Running N=5 (actual=5) x 200 reps...
  Running N=7 (actual=7) x 200 reps...
  Running N=10 (actual=8) x 200 reps...
  Running N=15 (actual=8) x 200 reps...
  Running N=20 (actual=8) x 200 reps...
Glasses sweep complete.


In [ ]:
# CELL 6: Run sweep for BEARD
print('=' * 60)
print('EXPERIMENT 2: BEARD subgroup calibration size sweep')
print(f'K={K_REPS} repetitions per N, MIN_GROUP_SIZE=5')
print('=' * 60)

N_VALUES_BEARD = [1, 2]   # only 2 beard samples exist — boundary case
GROUPS_BEARD   = [0, 1]   # 0=no beard, 1=beard

beard_results, max_beard = run_sweep(
    cal_results, test_results, beard_map, 'has_beard',
    GROUPS_BEARD, N_VALUES_BEARD,
    k_reps=K_REPS, min_group_size=5
)
beard_summary = summarise_sweep(beard_results, N_VALUES_BEARD, max_beard)
print('Beard sweep complete.')

EXPERIMENT 2: BEARD subgroup calibration size sweep
K=200 repetitions per N, MIN_GROUP_SIZE=5
Minority cal samples: 2  Majority cal samples: 48
  Running N=1 (actual=1) x 200 reps...
  Running N=2 (actual=2) x 200 reps...
Beard sweep complete.


In [ ]:
# Safety save — run before leaving overnight
import pickle
with open(os.path.join(SAVE_DIR, 'glasses_results_backup.pkl'), 'wb') as f:
    pickle.dump((glasses_results, glasses_summary, N_VALUES_GLASSES, max_glasses), f)
with open(os.path.join(SAVE_DIR, 'beard_results_backup.pkl'), 'wb') as f:
    pickle.dump((beard_results, beard_summary, N_VALUES_BEARD, max_beard), f)
print('Backup saved to Drive.')

Backup saved to Drive.


In [ ]:
# CELL 7: MIN_GROUP_SIZE sensitivity analysis
print('=' * 60)
print('EXPERIMENT 3: MIN_GROUP_SIZE sensitivity (glasses, all N)')
print('=' * 60)

MIN_SIZE_VALUES = [2, 3, 5, 7, 10]
sensitivity_results = {}

for mgs in MIN_SIZE_VALUES:
    print(f'  MIN_GROUP_SIZE={mgs}...')
    res, _ = run_sweep(
        cal_results, test_results, glasses_map, 'has_glasses',
        GROUPS_GLASSES, N_VALUES_GLASSES,
        k_reps=K_REPS, min_group_size=mgs
    )
    sensitivity_results[mgs] = summarise_sweep(res, N_VALUES_GLASSES, max_glasses)

print('Sensitivity analysis complete.')

EXPERIMENT 3: MIN_GROUP_SIZE sensitivity (glasses, all N)
  MIN_GROUP_SIZE=2...
Minority cal samples: 8  Majority cal samples: 42
  Running N=2 (actual=2) x 200 reps...
  Running N=3 (actual=3) x 200 reps...
  Running N=4 (actual=4) x 200 reps...
  Running N=5 (actual=5) x 200 reps...
  Running N=7 (actual=7) x 200 reps...
  Running N=10 (actual=8) x 200 reps...
  Running N=15 (actual=8) x 200 reps...
  Running N=20 (actual=8) x 200 reps...
  MIN_GROUP_SIZE=3...
Minority cal samples: 8  Majority cal samples: 42
  Running N=2 (actual=2) x 200 reps...
  Running N=3 (actual=3) x 200 reps...
  Running N=4 (actual=4) x 200 reps...
  Running N=5 (actual=5) x 200 reps...
  Running N=7 (actual=7) x 200 reps...
  Running N=10 (actual=8) x 200 reps...
  Running N=15 (actual=8) x 200 reps...
  Running N=20 (actual=8) x 200 reps...
  MIN_GROUP_SIZE=5...
Minority cal samples: 8  Majority cal samples: 42
  Running N=2 (actual=2) x 200 reps...
  Running N=3 (actual=3) x 200 reps...
  Running N=4 (act

In [ ]:
# CELL 8: Print results tables

def print_sweep_table(summary, n_values, max_minority, attr_label):
    print(f'\n=== {attr_label} — PICP Gap vs N (mean [95% CI]) ===')
    print(f'{"N":>4s}  {"CQR gap":>18s}  {"FUQ orig gap":>18s}  {"FUQ shrunk gap":>18s}')
    print('-' * 65)
    for n in n_values:
        marker = '*' if n > max_minority else ' '
        for method, label in [('cqr','CQR'), ('fuq_orig','FUQ orig'), ('fuq_shrunk','FUQ shrunk')]:
            s = summary[n][method]
            print(f'{n:>4d}{marker} {label:12s}  gap={s["gap_mean"]:.3f} [{s["gap_lo"]:.3f}, {s["gap_hi"]:.3f}]')
        print()
    print('* = N capped at actual minority size')

print_sweep_table(glasses_summary, N_VALUES_GLASSES, max_glasses, 'GLASSES')
print_sweep_table(beard_summary,   N_VALUES_BEARD,   max_beard,   'BEARD')

print('\n=== MIN_GROUP_SIZE sensitivity — mean PICP gap at each N ===')
print(f'{"N":>4s}  ' + '  '.join([f'MGS={m:>2d}' for m in MIN_SIZE_VALUES]))
print('-' * 55)
for n in N_VALUES_GLASSES:
    row = f'{n:>4d}  '
    for mgs in MIN_SIZE_VALUES:
        gap = sensitivity_results[mgs][n]['fuq_shrunk']['gap_mean']
        row += f'{gap:7.3f}  '
    print(row)


=== GLASSES — PICP Gap vs N (mean [95% CI]) ===
   N             CQR gap        FUQ orig gap      FUQ shrunk gap
-----------------------------------------------------------------
   2  CQR           gap=0.155 [0.155, 0.155]
   2  FUQ orig      gap=0.161 [0.155, 0.202]
   2  FUQ shrunk    gap=0.129 [0.030, 0.155]

   3  CQR           gap=0.140 [0.131, 0.155]
   3  FUQ orig      gap=0.188 [0.155, 0.304]
   3  FUQ shrunk    gap=0.133 [0.030, 0.280]

   4  CQR           gap=0.143 [0.131, 0.155]
   4  FUQ orig      gap=0.181 [0.155, 0.280]
   4  FUQ shrunk    gap=0.091 [0.030, 0.280]

   5  CQR           gap=0.146 [0.131, 0.155]
   5  FUQ orig      gap=0.170 [0.155, 0.280]
   5  FUQ shrunk    gap=0.068 [0.030, 0.131]

   7  CQR           gap=0.148 [0.107, 0.155]
   7  FUQ orig      gap=0.181 [0.155, 0.280]
   7  FUQ shrunk    gap=0.082 [0.030, 0.280]

  10* CQR           gap=0.155 [0.155, 0.155]
  10* FUQ orig      gap=0.155 [0.155, 0.155]
  10* FUQ shrunk    gap=0.030 [0.030, 0.030]

  15

In [ ]:
# CELL 9: MAIN FIGURE — PICP Gap vs Subgroup Calibration Size
# This is the paper's central figure.

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=False)

METHOD_STYLES = {
    'cqr':       {'color': '#888888', 'marker': 's', 'ls': '--',  'label': 'CQR'},
    'fuq_orig':  {'color': '#D85A30', 'marker': 'o', 'ls': '-',   'label': 'FUQ (original)'},
    'fuq_shrunk':{'color': '#1D6FA4', 'marker': '^', 'ls': '-',   'label': 'FUQ (shrinkage-aware)'},
}

def plot_sweep(ax, summary, n_values, max_minority, title):
    for method, style in METHOD_STYLES.items():
        ns, means, los, his = [], [], [], []
        for n in n_values:
            s = summary[n][method]
            ns.append(min(n, max_minority))
            means.append(s['gap_mean'])
            los.append(s['gap_lo'])
            his.append(s['gap_hi'])
        ns    = np.array(ns)
        means = np.array(means)
        los   = np.array(los)
        his   = np.array(his)

        ax.plot(ns, means, color=style['color'], marker=style['marker'],
                ls=style['ls'], linewidth=2, markersize=6, label=style['label'])
        ax.fill_between(ns, los, his, alpha=0.12, color=style['color'])

    ax.axhline(0.05, color='black', linestyle=':', linewidth=1, alpha=0.5,
               label='acceptable gap (0.05)')
    ax.set_xlabel('Minority subgroup calibration size (N)', fontsize=11)
    ax.set_ylabel('PICP gap |majority − minority|', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9, loc='upper right')
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plot_sweep(axes[0], glasses_summary, N_VALUES_GLASSES, max_glasses,
           'Glasses subgroup\n(sparse-but-present regime)')
plot_sweep(axes[1], beard_summary, N_VALUES_BEARD, max_beard,
           'Beard subgroup\n(extremely sparse boundary regime)')

plt.suptitle('PICP Gap vs Minority Subgroup Calibration Size\n'
             'Shaded regions show 95% CI across 200 random subsamples',
             fontsize=12, y=1.01)
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig_picp_gap_vs_n.png')
plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved: {path}')

Saved: /content/drive/MyDrive/avec2014_sparse_subgroup/fig_picp_gap_vs_n.png


In [ ]:
# CELL 10: Secondary figure — Overall PICP and MPIW vs N
# Shows that improving fairness does not hurt overall coverage or efficiency.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def plot_metric(ax, summary, n_values, max_minority, metric, ylabel, title, hline=None):
    for method, style in METHOD_STYLES.items():
        ns, means, los, his = [], [], [], []
        for n in n_values:
            s = summary[n][method]
            ns.append(min(n, max_minority))
            means.append(s[f'{metric}_mean'])
            los.append(s[f'{metric}_lo'])
            his.append(s[f'{metric}_hi'])
        ns    = np.array(ns)
        means = np.array(means)
        los   = np.array(los)
        his   = np.array(his)
        ax.plot(ns, means, color=style['color'], marker=style['marker'],
                ls=style['ls'], linewidth=2, markersize=6, label=style['label'])
        ax.fill_between(ns, los, his, alpha=0.12, color=style['color'])
    if hline is not None:
        ax.axhline(hline, color='black', linestyle=':', linewidth=1, alpha=0.5,
                   label=f'target ({hline})')
    ax.set_xlabel('Minority subgroup calibration size (N)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plot_metric(axes[0], glasses_summary, N_VALUES_GLASSES, max_glasses,
            'picp_all', 'Overall PICP', 'Overall coverage vs N (glasses)', hline=0.90)
plot_metric(axes[1], glasses_summary, N_VALUES_GLASSES, max_glasses,
            'mpiw_all', 'MPIW', 'Interval width vs N (glasses)')

plt.suptitle('Overall Coverage and Interval Width vs Minority Calibration Size',
             fontsize=12, y=1.01)
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig_picp_mpiw_vs_n.png')
plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved: {path}')

Saved: /content/drive/MyDrive/avec2014_sparse_subgroup/fig_picp_mpiw_vs_n.png


In [ ]:
# CELL 11: MIN_GROUP_SIZE sensitivity figure

fig, ax = plt.subplots(figsize=(9, 5))

mgs_colors = ['#1D6FA4', '#2E9E6B', '#D85A30', '#8B4CA8', '#C4A000']
for mgs, color in zip(MIN_SIZE_VALUES, mgs_colors):
    ns, means, los, his = [], [], [], []
    for n in N_VALUES_GLASSES:
        s = sensitivity_results[mgs][n]['fuq_shrunk']
        ns.append(min(n, max_glasses))
        means.append(s['gap_mean'])
        los.append(s['gap_lo'])
        his.append(s['gap_hi'])
    ns    = np.array(ns)
    means = np.array(means)
    ax.plot(ns, means, color=color, marker='o', linewidth=1.8,
            markersize=5, label=f'MIN_GROUP_SIZE={mgs}')
    ax.fill_between(ns, los, his, alpha=0.08, color=color)

ax.axhline(0.05, color='black', linestyle=':', linewidth=1, alpha=0.5,
           label='acceptable gap (0.05)')
ax.set_xlabel('Minority subgroup calibration size (N)', fontsize=11)
ax.set_ylabel('PICP gap (Shrinkage-FUQ)', fontsize=11)
ax.set_title('Sensitivity of Shrinkage-FUQ to MIN_GROUP_SIZE hyperparameter\n'
             '(glasses subgroup, shaded = 95% CI)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(bottom=0)

plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig_min_group_size_sensitivity.png')
plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved: {path}')

Saved: /content/drive/MyDrive/avec2014_sparse_subgroup/fig_min_group_size_sensitivity.png


In [ ]:
# CELL 12: Subgroup PICP figure — minority group only
# Shows PICP of the minority group itself vs N for all three methods

fig, ax = plt.subplots(figsize=(9, 5))

for method, style in METHOD_STYLES.items():
    ns, means, los, his = [], [], [], []
    for n in N_VALUES_GLASSES:
        s = glasses_summary[n][method]
        ns.append(min(n, max_glasses))
        means.append(s['picp_min_mean'])
        los.append(s['picp_min_lo'])
        his.append(s['picp_min_hi'])
    ns    = np.array(ns)
    means = np.array(means)
    los   = np.array(los)
    his   = np.array(his)
    ax.plot(ns, means, color=style['color'], marker=style['marker'],
            ls=style['ls'], linewidth=2, markersize=6, label=style['label'])
    ax.fill_between(ns, los, his, alpha=0.12, color=style['color'])

ax.axhline(TARGET, color='black', linestyle=':', linewidth=1.2,
           alpha=0.6, label=f'target PICP ({TARGET})')
ax.set_xlabel('Glasses subgroup calibration size (N)', fontsize=11)
ax.set_ylabel('Glasses subgroup PICP', fontsize=11)
ax.set_title('Minority subgroup PICP vs calibration size\n'
             '(glasses group only, shaded = 95% CI across 200 subsamples)', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig_subgroup_picp_vs_n.png')
plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved: {path}')

Saved: /content/drive/MyDrive/avec2014_sparse_subgroup/fig_subgroup_picp_vs_n.png


In [ ]:
# CELL 13: Save all numerical results to CSV for reproducibility

def summary_to_df(summary, n_values, max_minority, attr_label):
    rows = []
    for n in n_values:
        for method in ['cqr', 'fuq_orig', 'fuq_shrunk']:
            s = summary[n][method]
            rows.append({
                'attribute': attr_label,
                'N_requested': n,
                'N_actual': min(n, max_minority),
                'method': method,
                'picp_all_mean':   s['picp_all_mean'],
                'picp_all_lo':     s['picp_all_lo'],
                'picp_all_hi':     s['picp_all_hi'],
                'mpiw_mean':       s['mpiw_all_mean'],
                'mpiw_lo':         s['mpiw_all_lo'],
                'mpiw_hi':         s['mpiw_all_hi'],
                'picp_minority_mean': s['picp_min_mean'],
                'picp_minority_lo':   s['picp_min_lo'],
                'picp_minority_hi':   s['picp_min_hi'],
                'picp_majority_mean': s['picp_maj_mean'],
                'gap_mean':        s['gap_mean'],
                'gap_lo':          s['gap_lo'],
                'gap_hi':          s['gap_hi'],
            })
    return pd.DataFrame(rows)

glasses_df = summary_to_df(glasses_summary, N_VALUES_GLASSES, max_glasses, 'glasses')
beard_df   = summary_to_df(beard_summary,   N_VALUES_BEARD,   max_beard,   'beard')
all_df     = pd.concat([glasses_df, beard_df], ignore_index=True)
all_df.to_csv(os.path.join(SAVE_DIR, 'sweep_results.csv'), index=False)

# Sensitivity results
sens_rows = []
for mgs in MIN_SIZE_VALUES:
    for n in N_VALUES_GLASSES:
        s = sensitivity_results[mgs][n]['fuq_shrunk']
        sens_rows.append({
            'min_group_size': mgs, 'N': min(n, max_glasses),
            'gap_mean': s['gap_mean'], 'gap_lo': s['gap_lo'], 'gap_hi': s['gap_hi']
        })
pd.DataFrame(sens_rows).to_csv(os.path.join(SAVE_DIR, 'sensitivity_results.csv'), index=False)

print(f'All results saved to: {SAVE_DIR}')
print('Files:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e3
    print(f'  {f}  ({size:.0f} KB)')

All results saved to: /content/drive/MyDrive/avec2014_sparse_subgroup
Files:
  fig_min_group_size_sensitivity.png  (371 KB)
  fig_picp_gap_vs_n.png  (428 KB)
  fig_picp_mpiw_vs_n.png  (454 KB)
  fig_subgroup_picp_vs_n.png  (204 KB)
  sensitivity_results.csv  (3 KB)
  sweep_results.csv  (6 KB)
